In [1]:
import gradio as gr
import random
import json
import os
import re

from datetime import datetime


In [2]:
LEADERBOARD_FILE = "leaderboard.json"

RUMORS = [
    "The manager announced a secret meeting tomorrow.",
    "My friend is bringing coffee to the office.",
    "The birthday celebration will happen tomorrow.",
    "Our team is preparing a new project.",
    "The manager invited everyone for lunch.",
    "The teacher organized a surprise party.",
    "The director announced a new competition.",
    "The students planned a surprise celebration.",
    "The company launched a new mobile application.",
    "My colleague purchased a beautiful birthday gift."
]

REPLACEMENTS = {
    "office": ["school", "company", "restaurant"],
    "manager": ["teacher", "director", "captain"],
    "meeting": ["party", "event", "celebration"],
    "tomorrow": ["today", "Monday", "Sunday"],
    "coffee": ["tea", "juice", "milk"],
    "lunch": ["dinner", "breakfast", "snacks"],
    "secret": ["surprise", "mystery", "special"],
    "friend": ["colleague", "neighbor", "student"],
    "birthday": ["wedding", "festival", "celebration"],
    "project": ["competition", "presentation", "assignment"],
    "teacher": ["manager", "principal", "director"],
    "company": ["school", "office", "restaurant"],
    "party": ["meeting", "event", "festival"],
    "event": ["meeting", "celebration", "competition"],
    "school": ["office", "college", "company"],
    "director": ["manager", "teacher", "captain"],
    "celebration": ["party", "meeting", "festival"],
    "surprise": ["secret", "mystery", "special"],
    "competition": ["project", "event", "presentation"],
    "presentation": ["project", "competition", "assignment"],
    "colleague": ["friend", "neighbor", "student"],
    "restaurant": ["office", "school", "company"],
    "dinner": ["lunch", "breakfast", "snacks"],
    "festival": ["birthday", "wedding", "celebration"],
    "captain": ["manager", "director", "teacher"],
    "college": ["school", "office", "company"],
    "principal": ["teacher", "manager", "director"],
    "special": ["secret", "surprise", "mystery"],
    "assignment": ["project", "competition", "presentation"],
    "neighbor": ["friend", "colleague", "student"],
    "student": ["friend", "colleague", "neighbor"],
    "gift": ["present", "surprise", "prize"],
    "present": ["gift", "prize", "surprise"],
    "prize": ["gift", "present", "surprise"],
    "application": ["website", "software", "platform"],
    "mobile": ["computer", "digital", "online"],
    "beautiful": ["wonderful", "amazing", "special"]
}

DIFFICULTY_LEVELS = {
    "Easy": 1,
    "Medium": 2,
    "Hard": 3
}

MAX_AI_PLAYERS = 10
MIN_AI_PLAYERS = 5
MAX_HUMAN_PLAYERS = 4


In [3]:
class Player:

    def __init__(self, name, is_human=True):

        self.name = name
        self.is_human = is_human

    def change_rumor(self, message, difficulty):

        words = message.split()

        possible_indexes = []

        # Find words that can be replaced
        for index, word in enumerate(words):

            clean_word = word.lower().strip(".,!?")

            if clean_word in REPLACEMENTS:

                possible_indexes.append(index)

        # Shuffle available words
        random.shuffle(possible_indexes)

        # Number of words to change
        changes = DIFFICULTY_LEVELS[difficulty]

        for index in possible_indexes[:changes]:

            old_word = words[index]

            clean_word = old_word.lower().strip(".,!?")

            # Select replacement word
            new_word = random.choice(
                REPLACEMENTS[clean_word]
            )

            # Preserve punctuation
            punctuation = (
                old_word[-1]
                if old_word[-1] in ".,!?"
                else ""
            )

            words[index] = new_word + punctuation

        return " ".join(words)

    def __str__(self):

        return self.name


In [4]:
class Leaderboard:

    def __init__(self):

        self.records = self.load_scores()

    def load_scores(self):

        if os.path.exists(LEADERBOARD_FILE):

            try:

                with open(
                    LEADERBOARD_FILE, "r"
                ) as file:

                    data = json.load(file)

                    return data if isinstance(data, list) else []

            except (json.JSONDecodeError, OSError):

                return []

        return []

    def save_score(self, name, score):

        record = {

            "name": name,

            "score": score,

            "date": datetime.now().strftime(
                "%d-%m-%Y %H:%M"
            )
        }

        self.records.append(record)

        self.records.sort(
            key=lambda item: item["score"],
            reverse=True
        )

        # Keep top 10 scores
        self.records = self.records[:10]

        with open(
            LEADERBOARD_FILE, "w"
        ) as file:

            json.dump(
                self.records,
                file,
                indent=4
            )

    def get_leaderboard(self):

        if not self.records:

            return "No leaderboard records available."

        output = []

        for rank, record in enumerate(
            self.records, start=1
        ):

            output.append(
                f"{rank}. {record['name']} | "
                f"{record['score']} points | "
                f"{record.get('date', 'N/A')}"
            )

        return "\n".join(output)


In [5]:
class RumorMillGame:

    def __init__(self):

        self.leaderboard = Leaderboard()

        self.players = []

        self.original_message = ""
        self.current_message = ""

        self.current_player = 0

        self.current_round = 0
        self.total_rounds = 0

        self.total_score = 0

        self.difficulty = "Easy"

        self.captain_name = ""

        self.history = []

        self.game_over = False
        self.round_over = False

    # ----------------------------------
    # START NEW GAME
    # ----------------------------------

    def start_game(
        self,
        human_names,
        ai_count,
        difficulty,
        rounds
    ):

        names = [
            name.strip()
            for name in human_names.split(",")
            if name.strip()
        ]

        if not names:
            raise ValueError(
                "Enter at least one human player."
            )

        if len(names) > MAX_HUMAN_PLAYERS:
            raise ValueError(
                "Maximum 4 human players allowed."
            )

        if not 5 <= ai_count <= 10:
            raise ValueError(
                "AI players must be between 5 and 10."
            )

        if difficulty not in DIFFICULTY_LEVELS:
            raise ValueError("Invalid difficulty.")

        if not 1 <= rounds <= 5:
            raise ValueError(
                "Rounds must be between 1 and 5."
            )

        self.players = []

        # Create human players
        for name in names:

            self.players.append(
                Player(name, True)
            )

        # Create AI players
        for number in range(1, ai_count + 1):

            self.players.append(
                Player(f"AI-{number}", False)
            )

        self.captain_name = names[0]

        self.difficulty = difficulty

        self.total_rounds = rounds

        self.total_score = 0

        self.current_round = 0

        self.history = []

        self.game_over = False

        return self.start_round()

    # ----------------------------------
    # START ROUND
    # ----------------------------------

    def start_round(self):

        if self.current_round >= self.total_rounds:

            self.game_over = True

            return self.finish_game()

        self.current_round += 1

        self.current_player = 0

        self.round_over = False

        # Generate original message
        self.original_message = random.choice(RUMORS)

        self.current_message = self.original_message

        self.history.append(
            f"\n===== ROUND {self.current_round} ====="
        )

        self.history.append(
            f"ORIGINAL: {self.original_message}"
        )

        return self.advance_turns()

    # ----------------------------------
    # ADVANCE PLAYER TURNS
    # ----------------------------------

    def advance_turns(self):

        while self.current_player < len(self.players):

            player = self.players[self.current_player]

            # Stop when a human player's turn arrives
            if player.is_human:

                return {
                    "status": f"{player.name}'s Turn",
                    "message": self.current_message,
                    "input_required": True,
                    "completed": False
                }

            # AI automatically changes the rumor
            self.current_message = player.change_rumor(
                self.current_message,
                self.difficulty
            )

            self.history.append(
                f"{player.name}: {self.current_message}"
            )

            self.current_player += 1

        return self.finish_round()

    # ----------------------------------
    # HUMAN PLAYER SUBMISSION
    # ----------------------------------

    def submit_human_rumor(self, new_message):

        if self.game_over or self.round_over:
            raise ValueError("No active player turn.")

        if self.current_player >= len(self.players):
            raise ValueError("No player turn available.")

        player = self.players[self.current_player]

        if not player.is_human:
            raise ValueError("It is not a human turn.")

        if not new_message or not new_message.strip():

            raise ValueError(
                "Enter a rumor before submitting."
            )

        self.current_message = new_message.strip()

        self.history.append(
            f"{player.name}: {self.current_message}"
        )

        self.current_player += 1

        # Process following players
        return self.advance_turns()

    # ----------------------------------
    # CALCULATE ACCURACY
    # ----------------------------------

    def calculate_accuracy(self):

        original_words = re.findall(
            r"\b\w+\b",
            self.original_message.lower()
        )

        final_words = re.findall(
            r"\b\w+\b",
            self.current_message.lower()
        )

        if not original_words:
            return 0

        correct = sum(
            1
            for original, final in zip(
                original_words,
                final_words
            )
            if original == final
        )

        accuracy = (
            correct / len(original_words)
        ) * 100

        return round(accuracy, 2)

    # ----------------------------------
    # FINISH ROUND
    # ----------------------------------

    def finish_round(self):

        accuracy = self.calculate_accuracy()

        # Points based on accuracy
        points = round(accuracy)

        self.total_score += points

        self.round_over = True

        self.history.append(
            f"FINAL RUMOR: {self.current_message}"
        )

        self.history.append(
            f"ACCURACY: {accuracy}%"
        )

        self.history.append(
            f"POINTS: {points}"
        )

        return {

            "status": f"Round {self.current_round} Completed",

            "message": self.current_message,

            "input_required": False,

            "completed": True,

            "original": self.original_message,

            "accuracy": accuracy,

            "points": points,

            "total_score": self.total_score
        }

    # ----------------------------------
    # FINISH GAME
    # ----------------------------------

    def finish_game(self):

        self.game_over = True

        self.leaderboard.save_score(
            self.captain_name,
            self.total_score
        )

        return {

            "status": "GAME OVER",

            "message": (
                f"Congratulations {self.captain_name}!\n"
                f"Final Score: {self.total_score}"
            ),

            "input_required": False,

            "completed": True,

            "total_score": self.total_score
        }

    # ----------------------------------
    # LEADERBOARD
    # ----------------------------------

    def get_leaderboard(self):

        return self.leaderboard.get_leaderboard()

    # ----------------------------------
    # GAME HISTORY
    # ----------------------------------

    def get_history(self):

        return "\n".join(self.history)


In [7]:
def finish_round(self):

    accuracy = self.calculate_accuracy()

    points = round(accuracy)

    self.total_score += points

    self.round_over = True

    self.history.append(
        f"FINAL RUMOR: {self.current_message}"
    )

    self.history.append(
        f"ACCURACY: {accuracy}%"
    )

    self.history.append(
        f"POINTS: {points}"
    )

    # Automatically save after final round
    if self.current_round >= self.total_rounds:

        self.game_over = True

        self.leaderboard.save_score(
            self.captain_name,
            self.total_score
        )

        self.history.append(
            "GAME COMPLETED!"
        )

        status = "GAME OVER - All rounds completed"

    else:

        status = f"Round {self.current_round} Completed"

    return {

        "status": status,

        "message": self.current_message,

        "input_required": False,

        "completed": True,

        "original": self.original_message,

        "accuracy": accuracy,

        "points": points,

        "total_score": self.total_score
    }


In [10]:
def create_game():

    return RumorMillGame()


def start_new_game(
    names,
    ai_count,
    difficulty,
    rounds
):

    game = create_game()

    try:

        result = game.start_game(
            names,
            int(ai_count),
            difficulty,
            int(rounds)
        )

        status = result["status"]

        rumor = result["message"]

        history = game.get_history()

        if result["completed"]:

            status += "\n\n"
            status += (
                f"Original: {result.get('original', '')}\n"
            )

            status += (
                f"Accuracy: {result.get('accuracy', 0)}%\n"
            )

            status += (
                f"Points: {result.get('points', 0)}"
            )

        return (

            game,

            status,

            rumor,

            gr.update(
                visible=result["input_required"],
                value=""
            ),

            gr.update(
                interactive=result["input_required"]
            ),

            gr.update(
                visible=result["completed"],
                interactive=not game.game_over
            ),

            history,

            f"Total Score: {game.total_score}"

        )

    except (ValueError, TypeError) as error:

        return (

            None,

            f"Error: {error}",

            "",

            gr.update(visible=False),

            gr.update(interactive=False),

            gr.update(visible=False),

            "",

            "Total Score: 0"

        )


def pass_rumor(game, new_message):

    if game is None:

        return (
            None,
            "Start a new game first.",
            "",
            gr.update(visible=False),
            gr.update(interactive=False),
            gr.update(visible=False),
            "",
            "Total Score: 0"
        )

    try:

        result = game.submit_human_rumor(
            new_message
        )

        status = result["status"]

        if result["completed"]:

            status += "\n\n"
            status += (
                f"Original: {result['original']}\n"
            )

            status += (
                f"Accuracy: {result['accuracy']}%\n"
            )

            status += (
                f"Points: {result['points']}"
            )

        return (

            game,

            status,

            result["message"],

            gr.update(
                visible=result["input_required"],
                value=""
            ),

            gr.update(
                interactive=result["input_required"]
            ),

            gr.update(
                visible=result["completed"],
                interactive=not game.game_over
            ),

            game.get_history(),

            f"Total Score: {game.total_score}"

        )

    except ValueError as error:

        return (

            game,

            f"Error: {error}",

            game.current_message,

            gr.update(visible=True),

            gr.update(interactive=True),

            gr.update(visible=False),

            game.get_history(),

            f"Total Score: {game.total_score}"

        )


def next_round(game):

    if game is None:

        return (
            None,
            "Start a new game first.",
            "",
            gr.update(visible=False),
            gr.update(interactive=False),
            gr.update(visible=False),
            "",
            "Total Score: 0"
        )

    if game.game_over:

        return (
            game,
            "Game already completed. Start a new game.",
            "",
            gr.update(visible=False),
            gr.update(interactive=False),
            gr.update(visible=False),
            game.get_history(),
            f"Total Score: {game.total_score}"
        )

    result = game.start_round()

    status = result["status"]

    if result["completed"]:

        status += (
            f"\nAccuracy: {result.get('accuracy', 0)}%"
        )

    return (

        game,

        status,

        result["message"],

        gr.update(
            visible=result["input_required"],
            value=""
        ),

        gr.update(
            interactive=result["input_required"]
        ),

        gr.update(
            visible=result["completed"],
            interactive=not game.game_over
        ),

        game.get_history(),

        f"Total Score: {game.total_score}"

    )


def show_leaderboard():

    board = Leaderboard()

    return board.get_leaderboard()


# --------------------------------------
# BUILD GRADIO APPLICATION
# --------------------------------------

with gr.Blocks(
    theme=gr.themes.Soft(),
    title="Rumor Mill Game"
) as demo:

    game_state = gr.State(None)

    gr.Markdown(
        """
        # 🎮 RUMOR MILL GAME

        ### Pass the message. Change the story. Become the Rumor Master!

        Welcome to the interactive Python Rumor Mill Game.
        """
    )

    # Game settings
    with gr.Accordion(
        "⚙️ Game Settings",
        open=True
    ):

        with gr.Row():

            player_names = gr.Textbox(
                label="Human Player Names",
                value="Alex, Sam",
                placeholder="Enter names separated by commas"
            )

            ai_count = gr.Slider(
                minimum=5,
                maximum=10,
                step=1,
                value=5,
                label="AI Characters"
            )

        with gr.Row():

            difficulty = gr.Dropdown(
                choices=["Easy", "Medium", "Hard"],
                value="Easy",
                label="Difficulty"
            )

            rounds = gr.Dropdown(
                choices=[1, 2, 3, 4, 5],
                value=3,
                label="Number of Rounds"
            )

        start_button = gr.Button(
            "🚀 START GAME",
            variant="primary"
        )

    gr.Markdown("---")

    # Game status
    status_output = gr.Textbox(
        label="Game Status",
        interactive=False
    )

    score_output = gr.Markdown(
        "**Total Score: 0**"
    )

    # Rumor area
    with gr.Group():

        rumor_output = gr.Textbox(
            label="Current Rumor",
            lines=3,
            interactive=False
        )

        human_input = gr.Textbox(
            label="Change the Rumor",
            placeholder="Type your changed rumor here...",
            lines=3,
            visible=False
        )

        pass_button = gr.Button(
            "📨 PASS RUMOR",
            variant="primary",
            interactive=False
        )

        next_button = gr.Button(
            "➡️ NEXT ROUND",
            variant="secondary",
            visible=False
        )

    # History
    with gr.Accordion(
        "📜 Game History",
        open=True
    ):

        history_output = gr.Textbox(
            label="Rumor History",
            lines=12,
            interactive=False
        )

    # Leaderboard
    gr.Markdown("---")

    with gr.Accordion(
        "🏆 LEADERBOARD",
        open=False
    ):

        leaderboard_output = gr.Textbox(
            label="Top 10 Players",
            lines=12,
            interactive=False
        )

        leaderboard_button = gr.Button(
            "🔄 REFRESH LEADERBOARD"
        )

    # ----------------------------------
    # EVENT HANDLERS
    # ----------------------------------

    game_outputs = [

        game_state,
        status_output,
        rumor_output,
        human_input,
        pass_button,
        next_button,
        history_output,
        score_output

    ]

    start_button.click(

        fn=start_new_game,

        inputs=[
            player_names,
            ai_count,
            difficulty,
            rounds
        ],

        outputs=game_outputs

    )

    pass_button.click(

        fn=pass_rumor,

        inputs=[
            game_state,
            human_input
        ],

        outputs=game_outputs

    )

    next_button.click(

        fn=next_round,

        inputs=[game_state],

        outputs=game_outputs

    )

    leaderboard_button.click(

        fn=show_leaderboard,

        inputs=[],

        outputs=[leaderboard_output]

    )


# --------------------------------------
# LAUNCH APPLICATION
# --------------------------------------

demo.launch(share=False)


C:\Users\sunil\AppData\Local\Temp\ipykernel_22880\2666039763.py:263: UserWarning: The parameters have been moved from the Blocks constructor to the launch() method in Gradio 6.0: theme. Please pass these parameters to launch() instead.
  with gr.Blocks(


* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.
